# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md`. Đã triển khai Checkpoint 1 và mã E0 (Checkpoint 2); chạy từng mốc và dừng để kiểm chứng/commit.


## 0. Môi trường (Colab T4)

### Checkpoint 1 — Môi trường, dữ liệu và nhãn
Chọn **T4 GPU** trong Runtime → Change runtime type. Chạy lần lượt các cell đến mốc dừng Checkpoint 1.
Mã nguồn được clone vào `/content`; dữ liệu và trọng số không đưa vào Git. Drive giữ bản dự phòng khi Colab ngắt.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Nituv05/K4-Track-4-Day10-VuThuongTin-2A202602955-VisionLanguageModel.git"
REPO_DIR = Path("/content/road-damage-lab")
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
# Nếu notebook/mã Checkpoint 1 chưa có trong clone: push từ máy trước,
# rồi chạy `git pull --ff-only` ở Colab và chạy lại cell này.
if not (REPO_DIR / "student/data.py").exists():
    raise RuntimeError("Push mã Checkpoint 1 lên GitHub rồi cập nhật clone Colab trước khi chạy.")
subprocess.run(["nvidia-smi", "-L"], check=True)


In [ ]:
%pip -q install -r requirements-colab.txt pytest
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Hãy chọn runtime T4 và chạy lại.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run(["python", "-m", "pytest", "-q", "tests", "student/tests"], check=True)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/LabDay10_VisionLanguageModel")
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Nơi lưu checkpoint:", DRIVE_RUN_DIR)


In [ ]:
# Chọn đúng một nguồn. ZIP/SOURCE là đường dẫn trên Colab hoặc Google Drive.
DATA_MODE = "kaggle"  # "kaggle", "zip", hoặc "source"
DATA_ZIP = ""         # Ví dụ: /content/drive/MyDrive/road-damage.zip
DATA_SOURCE = ""      # Thư mục có images/ và labels-YOLO/

if DATA_MODE == "kaggle":
    from google.colab import userdata
    try:
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        raise RuntimeError("Thêm KAGGLE_USERNAME và KAGGLE_KEY vào Colab Secrets, bật quyền notebook; hoặc chọn zip/source.") from None
    if not os.environ.get("KAGGLE_USERNAME") or not os.environ.get("KAGGLE_KEY"):
        raise RuntimeError("Colab Secrets Kaggle đang trống.")
# Không in giá trị Secrets và không ghi chúng vào notebook.


## 1. Dữ liệu: tải và chia split

In [ ]:
if DATA_MODE == "kaggle":
    data_args = ["--kaggle"]
elif DATA_MODE == "zip" and Path(DATA_ZIP).is_file():
    data_args = ["--zip", DATA_ZIP]
elif DATA_MODE == "source" and Path(DATA_SOURCE).is_dir():
    data_args = ["--source", DATA_SOURCE]
else:
    raise ValueError("Kiểm tra DATA_MODE và đường dẫn nguồn dữ liệu.")
subprocess.run(["python", "scripts/prepare_data.py", *data_args], check=True)


In [ ]:
from student.data import write_checkpoint1
import json

cp1_summary = write_checkpoint1()
print(json.dumps(cp1_summary, ensure_ascii=False, indent=2))


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from student.data import PROMPT, objects_to_answer, inspect_splits
from IPython.display import display
from PIL import Image

records, _ = inspect_splits()
example = records["train"][0]
print("Prompt:", PROMPT)
print("Image:", example["image_id"])
print("Answer:", objects_to_answer(example["objects"]))
# Trái: box YOLO gốc; phải: JSON 0–1000 quy ngược về pixel.
for image_path in sorted(Path("work/checkpoint1").glob("labels_*.png")):
    with Image.open(image_path) as image:
        display(image.copy())


In [ ]:
import shutil
from google.colab import files

backup_dir = DRIVE_RUN_DIR / "checkpoint1"
shutil.copytree("work/checkpoint1", backup_dir, dirs_exist_ok=True)
shutil.copy2("data/splits/meta.json", backup_dir / "split_meta.json")
archive = shutil.make_archive("/content/checkpoint1", "zip", backup_dir)
print("Đã sao lưu:", backup_dir)
files.download(archive)


### DỪNG — Checkpoint 1
Kiểm tra sáu ảnh: box ở hai phía phải khớp (sai số làm tròn tối đa 0,32 pixel theo chiều rộng).
Gửi `checkpoint1.zip` và log pytest để kiểm chứng dữ liệu thật. Đây là ảnh **nhãn**, chưa phải sáu ảnh dự đoán đúng/sai cần cho báo cáo cuối.
Nếu đạt, tự commit/push Checkpoint 1. Chỉ tiếp tục E0 sau khi đã xác nhận mốc này.
Các phần phía dưới sẽ được hoàn thiện ở Checkpoint tiếp theo.


## 3. E0: zero-shot

### Checkpoint 2 — E0 zero-shot
Push mã Checkpoint 2 trước khi chạy trên Colab. Nếu clone đã tồn tại, cập nhật bằng `git pull --ff-only` trong thư mục repo, rồi restart runtime và chạy lại phần môi trường nếu đã import mã cũ.
Chạy thử **40 ảnh val**; đọc output và kiểm tra thang toạ độ trước khi chấm toàn bộ test. E0 không huấn luyện.
Output thô và metadata được sao lưu theo từng batch lên Drive; có thể tiếp tục cùng cấu hình sau khi runtime ngắt.


In [ ]:
import json
import shutil
from pathlib import Path
from importlib.metadata import version
import yaml
import torch
from transformers import set_seed
from student.data import inspect_splits
from student.inference import (load_model_processor, run_inference,
                               prediction_diagnostics, score_predictions, write_json)

if not Path("student/inference.py").exists():
    raise RuntimeError("Cần push/cập nhật mã Checkpoint 2 vào clone Colab.")
set_seed(42)
e0_config = yaml.safe_load(Path("configs/e0.yaml").read_text())
E0_BATCH_SIZE = 4
E0_RUN_LABEL = "e0"  # Nếu đổi cấu hình sau lần chạy dở, chọn nhãn mới, ví dụ e0_b1.
E0_MAX_NEW_TOKENS = e0_config["max_new_tokens"]
if E0_BATCH_SIZE < 1 or Path(E0_RUN_LABEL).name != E0_RUN_LABEL or not E0_RUN_LABEL:
    raise ValueError("Batch size phải dương và nhãn chạy phải là tên file đơn.")
e0_records, e0_split_summary = inspect_splits(
    expected_images={"train": 1422, "val": 276, "test": 311})
e0_backup_dir = DRIVE_RUN_DIR / "checkpoint2" / E0_RUN_LABEL
e0_backup_dir.mkdir(parents=True, exist_ok=True)

def backup_e0_file(path):
    path = Path(path)
    destination = e0_backup_dir / path.name
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    shutil.copy2(path, temporary)
    temporary.replace(destination)

def backup_e0_progress(pred_path, run_path):
    # Run metadata first: even an interrupted copy leaves a verifiable prefix.
    backup_e0_file(run_path)
    backup_e0_file(pred_path)

def restore_e0_output(pred_path):
    pred_path = Path(pred_path)
    run_path = pred_path.with_suffix(".run.json")
    saved_pred = e0_backup_dir / pred_path.name
    saved_run = e0_backup_dir / run_path.name
    if not pred_path.exists() and saved_pred.exists():
        if not saved_run.exists():
            raise RuntimeError("Bản dự phòng thiếu run metadata; không tiếp tục output này.")
        pred_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(saved_run, run_path)
        shutil.copy2(saved_pred, pred_path)
        print("Khôi phục từ Drive:", pred_path)

# Tránh giữ hai model trong VRAM nếu chạy lại cell.
if "e0_model" in globals():
    del e0_model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
e0_model, e0_processor, e0_model_audit = load_model_processor(e0_config["model_id"])
write_json("work/e0.model.json", {
    **e0_model_audit,
    "gpu": torch.cuda.get_device_name(0),
    "versions": {name: version(name) for name in
                 ("torch", "torchvision", "transformers", "bitsandbytes", "accelerate")},
})
print(json.dumps(e0_model_audit, indent=2))
backup_e0_file("work/e0.model.json")


In [ ]:
# Smoke test: 40 ảnh đầu của val, không tạo evidence test.
e0_val_rows = e0_records["val"][:40]
e0_val_path = Path("work") / f"{E0_RUN_LABEL}_val.jsonl"
restore_e0_output(e0_val_path)
e0_val_predictions = run_inference(
    e0_model, e0_processor, e0_val_rows, "data", e0_val_path,
    e0_split_summary["dataset_version"], "val", model_id=e0_config["model_id"],
    batch_size=E0_BATCH_SIZE, max_new_tokens=E0_MAX_NEW_TOKENS,
    progress_callback=backup_e0_progress,
)
e0_val_diagnostics = prediction_diagnostics(e0_val_rows, e0_val_predictions)
write_json("work/e0_val.diagnostics.json", e0_val_diagnostics)
backup_e0_file("work/e0_val.diagnostics.json")
print(json.dumps(e0_val_diagnostics, ensure_ascii=False, indent=2))


**Đọc output val trước khi tiếp tục:** có đủ một dòng mỗi ảnh, model chỉ được yêu cầu trả JSON, toạ độ 0–1000 và tên lớp đúng. JSON của E0 có thể kém; ghi nhận lỗi thật, không sửa output thô.
Nếu CUDA hết bộ nhớ: restart runtime, đặt `E0_BATCH_SIZE = 1`, chọn nhãn mới như `e0_b1`, rồi chạy lại phần E0. Nếu output bị cắt, điều chỉnh `E0_MAX_NEW_TOKENS` trên val và chọn nhãn chạy mới trước khi chạy test. Không đổi prompt/cấu hình dựa trên kết quả test.


In [ ]:
# Sau khi kiểm tra val, suy luận toàn bộ 311 ảnh test một lần.
e0_test_path = Path("work") / f"{E0_RUN_LABEL}.jsonl"
restore_e0_output(e0_test_path)
torch.cuda.reset_peak_memory_stats()
e0_test_predictions = run_inference(
    e0_model, e0_processor, e0_records["test"], "data", e0_test_path,
    e0_split_summary["dataset_version"], "test", model_id=e0_config["model_id"],
    batch_size=E0_BATCH_SIZE, max_new_tokens=E0_MAX_NEW_TOKENS,
    progress_callback=backup_e0_progress,
)
print("Peak inference VRAM GB:", torch.cuda.max_memory_allocated() / 1024**3)


In [ ]:
# Gọi scorer gốc, chỉ tạo evidence khi đủ toàn bộ ID test.
e0_evidence = score_predictions(
    e0_test_path, "report/evidence/e0.json", name="E0", meta={"trainable_params": 0})
e0_test_diagnostics = prediction_diagnostics(e0_records["test"], e0_test_predictions)
write_json("work/e0_test.diagnostics.json", e0_test_diagnostics)
backup_e0_file("report/evidence/e0.json")
backup_e0_file("work/e0_test.diagnostics.json")
backup_e0_file("data/splits/meta.json")
print("parse_valid_rate:", e0_evidence["parse_valid_rate"])
print("F1:", e0_evidence["micro"]["f1"], "CI95:", e0_evidence["f1_ci95"])
print("Triệu chứng output:", e0_test_diagnostics["symptoms_per_image"])
print("5 output thô:")
for example in e0_test_diagnostics["raw_examples"]:
    print(example["image_id"], "\n", example["raw"], "\n")


In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/checkpoint2", "zip", e0_backup_dir)
files.download(archive)
print("Bằng chứng để đưa về repo:", e0_backup_dir / "e0.json")


### DỪNG — Checkpoint 2
Gửi `checkpoint2.zip` và log để kiểm tra E0. Chỉ đưa file chấm thật `e0.json` vào `report/evidence/`; không commit predictions, dữ liệu hoặc trọng số.
Ghi nhận hai lỗi phổ biến bằng việc đối chiếu output với ảnh/GT. Các bộ đếm triệu chứng không tự kết luận sai thang, sai lớp hay bịa box.
Sau khi kiểm chứng, tự commit/push Checkpoint 2 rồi tiếp tục E1. Các phần dưới chưa được triển khai.


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9